# LeetCode 1408 — String Matching in an Array bằng Python

**Link đề:** https://leetcode.com/problems/string-matching-in-an-array/
**Độ khó:** Easy · **Chủ đề:** Array, String, String Matching

```python
class Solution:
    def stringMatching(self, words: list[str]) -> list[str]:
        ...
```

## Mục tiêu học
1. Hiểu **substring** (chuỗi con liên tiếp) và toán tử `in` của Python trên chuỗi.
2. Viết lời giải vét cạn hai vòng lặp đúng (bỏ qua chính nó, không thêm trùng).
3. Cải tiến bằng nhận xét **độ dài**: chỉ chuỗi dài hơn mới chứa được chuỗi ngắn → sắp theo độ dài.
4. Mẹo "**nối tất cả** rồi đếm số lần xuất hiện", và tự cài **KMP** để hiểu `in` hoạt động thế nào.
5. Trace, Time/Space, lỗi hay gặp, bài tập biến thể.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Cho mảng chuỗi `words`. Trả về **mọi** chuỗi trong `words` là **substring** (chuỗi con) của **một chuỗi khác** trong `words`. Thứ tự trả về **tùy ý**.

| `words` | Output | Giải thích |
|---|---|---|
| `["mass","as","hero","superhero"]` | `["as","hero"]` | `"as"` ⊂ `"mass"`, `"hero"` ⊂ `"superhero"` |
| `["leetcode","et","code"]` | `["et","code"]` | cả hai nằm trong `"leetcode"` |
| `["blue","green","bu"]` | `[]` | `"bu"` **không** liên tiếp trong `"blue"` |

**Constraints:** `1 <= words.length <= 100`, `1 <= words[i].length <= 30`, chỉ chữ thường, các chuỗi **đôi một khác nhau**.

Vì thứ tự tùy ý, khi test ta so sánh `sorted(kết_quả)` với `sorted(kỳ_vọng)`.

In [1]:
examples = [
    (["mass", "as", "hero", "superhero"], ["as", "hero"]),
    (["leetcode", "et", "code"], ["et", "code"]),
    (["blue", "green", "bu"], []),
]
for words, expected in examples:
    print(words, "-> kỳ vọng", expected)

['mass', 'as', 'hero', 'superhero'] -> kỳ vọng ['as', 'hero']
['leetcode', 'et', 'code'] -> kỳ vọng ['et', 'code']
['blue', 'green', 'bu'] -> kỳ vọng []


## 2. Nền tảng Python cần biết

### 2.1. Substring vs subsequence

- **Substring** = đoạn **liên tiếp**: `"as"` là substring của `"mass"` (vị trí 1–2).
- **Subsequence** = giữ thứ tự nhưng **được nhảy cóc**: `"bu"` là subsequence của `"blue"` (b, u) nhưng **không** phải substring.

Bài này hỏi **substring**.

In [2]:
s = "mass"
print("Các substring độ dài 2 của 'mass':", [s[i:i + 2] for i in range(len(s) - 1)])
print("'bu' có phải substring của 'blue'?", "bu" in "blue")

Các substring độ dài 2 của 'mass': ['ma', 'as', 'ss']
'bu' có phải substring của 'blue'? False


### 2.2. Toán tử `in` trên chuỗi và `str.find`

- `a in b` → `True` nếu `a` là substring của `b`.
- `b.find(a)` → vị trí **đầu tiên** của `a` trong `b`, hoặc `-1` nếu không có.
- Chuỗi rỗng `""` là substring của mọi chuỗi (đề không có chuỗi rỗng, nhưng nên biết).

In [3]:
print("as" in "mass", "hero" in "superhero", "bu" in "blue")
print("'mass'.find('as') =", "mass".find("as"))
print("'blue'.find('bu') =", "blue".find("bu"))
print("'' in 'abc' =", "" in "abc")

True True False
'mass'.find('as') = 1
'blue'.find('bu') = -1
'' in 'abc' = True


### 2.3. Slicing `s[i:j]` và so sánh chuỗi

`s[i:j]` lấy ký tự từ chỉ số `i` tới `j − 1`. Ta có thể tự kiểm tra substring "bằng tay": thử mọi vị trí bắt đầu `i` và so `s[i:i+len(p)] == p`.

In [4]:
def is_substring_manual(p: str, s: str) -> bool:
    for i in range(len(s) - len(p) + 1):
        if s[i:i + len(p)] == p:
            return True
    return False

for p, s in [("as", "mass"), ("bu", "blue"), ("hero", "superhero"), ("abc", "ab")]:
    print(f"{p!r} trong {s!r}? manual={is_substring_manual(p, s)}, in={p in s}")

'as' trong 'mass'? manual=True, in=True
'bu' trong 'blue'? manual=False, in=False
'hero' trong 'superhero'? manual=True, in=True
'abc' trong 'ab'? manual=False, in=False


`range(len(s) - len(p) + 1)` là các vị trí bắt đầu hợp lệ. Nếu `p` dài hơn `s` thì `range` rỗng → `False` ngay.

### 2.4. Sắp xếp theo khóa: `sorted(words, key=len)`

`key=len` bảo `sorted` so sánh theo **độ dài** thay vì theo thứ tự từ điển.

In [5]:
words = ["mass", "as", "hero", "superhero"]
print("theo từ điển:", sorted(words))
print("theo độ dài :", sorted(words, key=len))

theo từ điển: ['as', 'hero', 'mass', 'superhero']
theo độ dài : ['as', 'mass', 'hero', 'superhero']


### 2.5. `str.join` và `str.count`

- `" ".join(words)` nối các chuỗi, chèn `" "` giữa chúng.
- `s.count(p)` đếm số lần `p` xuất hiện (**không chồng lấn**) trong `s`.

In [6]:
s = " ".join(["mass", "as", "hero", "superhero"])
print(repr(s))
print("count('as')   =", s.count("as"))
print("count('hero') =", s.count("hero"))
print("count('mass') =", s.count("mass"))
print("'aaaa'.count('aa') =", "aaaa".count("aa"), "(không chồng lấn)")

'mass as hero superhero'
count('as')   = 2
count('hero') = 2
count('mass') = 1
'aaaa'.count('aa') = 2 (không chồng lấn)


## 3. Cách trực giác (baseline) — mọi cặp `(i, j)`

Với mỗi từ `words[i]`, thử mọi từ **khác** `words[j]` (`j != i`). Nếu `words[i] in words[j]` thì thêm `words[i]` vào kết quả và **dừng** (để không thêm trùng).

In [7]:
def string_matching_bruteforce(words: list[str]) -> list[str]:
    result = []
    for i in range(len(words)):
        for j in range(len(words)):
            if i != j and words[i] in words[j]:
                result.append(words[i])
                break                      # đã đủ một "chuỗi cha", khỏi thêm trùng
    return result

for words, expected in examples:
    print(words, "->", string_matching_bruteforce(words), "| kỳ vọng", expected)

['mass', 'as', 'hero', 'superhero'] -> ['as', 'hero'] | kỳ vọng ['as', 'hero']
['leetcode', 'et', 'code'] -> ['et', 'code'] | kỳ vọng ['et', 'code']
['blue', 'green', 'bu'] -> [] | kỳ vọng []


### 3.1. Độ phức tạp của baseline — có thật sự là "nút thắt"?

Gọi `n` = số từ (≤ 100), `L` = độ dài tối đa (≤ 30). Có `n²` cặp; mỗi lần `in` tốn tối đa khoảng `L²` phép so sánh ký tự (trường hợp xấu) ⇒ `O(n² · L²)` ≈ `10^4 · 900 ≈ 10^7`. Với Python, `in` cài bằng C rất nhanh ⇒ baseline **đã đủ** cho constraints.

Vậy các cải tiến sau là để **học kỹ thuật**: giảm số cặp phải thử, và hiểu cách tìm substring hiệu quả.

In [8]:
pairs_checked = 0
words = ["mass", "as", "hero", "superhero"]
for i in range(len(words)):
    for j in range(len(words)):
        if i != j:
            pairs_checked += 1
print("baseline thử", pairs_checked, "cặp cho", len(words), "từ")

baseline thử 12 cặp cho 4 từ


## 4. Insight — chuỗi ngắn chỉ nằm được trong chuỗi **dài hơn**

Nếu `a` là substring của `b` và `a != b` (các từ đôi một khác nhau) thì `len(a) < len(b)`. Vậy:
1. Sắp `words` theo độ dài tăng dần.
2. Với từ ở vị trí `i`, chỉ cần thử các từ ở vị trí `j > i` (dài hơn hoặc bằng).

Số cặp giảm còn khoảng một nửa (`n(n−1)/2`), và không cần điều kiện `i != j`.

Hai từ **cùng độ dài** mà khác nhau thì không chứa nhau — nhưng thử thêm cũng không sai, chỉ tốn chút thời gian.

In [9]:
words = sorted(["mass", "as", "hero", "superhero"], key=len)
for i, w in enumerate(words):
    print(f"{w!r:<12} chỉ cần thử với: {words[i + 1:]}")

'as'         chỉ cần thử với: ['mass', 'hero', 'superhero']
'mass'       chỉ cần thử với: ['hero', 'superhero']
'hero'       chỉ cần thử với: ['superhero']
'superhero'  chỉ cần thử với: []


## 5. Lời giải chính — sắp theo độ dài, giảng từng đoạn

**Đoạn 1:** `ws = sorted(words, key=len)`.

**Đoạn 2:** với mỗi `i`, duyệt `j` từ `i + 1` tới hết.

**Đoạn 3:** nếu `ws[i] in ws[j]` thì thêm `ws[i]` và `break`.

Kết quả theo thứ tự độ dài tăng dần — đề cho phép thứ tự bất kỳ.

In [10]:
def string_matching_sorted(words: list[str]) -> list[str]:
    ws = sorted(words, key=len)              # Đoạn 1
    result = []
    for i in range(len(ws)):
        for j in range(i + 1, len(ws)):      # Đoạn 2
            if ws[i] in ws[j]:               # Đoạn 3
                result.append(ws[i])
                break
    return result

for words, expected in examples:
    got = string_matching_sorted(words)
    print(words, "->", got, "| đúng (bỏ qua thứ tự)?", sorted(got) == sorted(expected))

['mass', 'as', 'hero', 'superhero'] -> ['as', 'hero'] | đúng (bỏ qua thứ tự)? True
['leetcode', 'et', 'code'] -> ['et', 'code'] | đúng (bỏ qua thứ tự)? True
['blue', 'green', 'bu'] -> [] | đúng (bỏ qua thứ tự)? True


### 5.1. Bản gọn bằng `any` và list comprehension

`any(điều_kiện for ...)` trả `True` ngay khi gặp phần tử đầu tiên thỏa (dừng sớm như `break`).

In [11]:
def string_matching_any(words: list[str]) -> list[str]:
    return [w for w in words if any(w != other and w in other for other in words)]

print(any(x > 3 for x in [1, 5, 2]), any(x > 9 for x in [1, 5, 2]))
for words, expected in examples:
    print(words, "->", string_matching_any(words))

True False
['mass', 'as', 'hero', 'superhero'] -> ['as', 'hero']
['leetcode', 'et', 'code'] -> ['et', 'code']
['blue', 'green', 'bu'] -> []


## 6. Cách 3 — nối tất cả, đếm số lần xuất hiện

Nối `s = " ".join(words)`. Mỗi từ `w` chắc chắn xuất hiện **ít nhất 1 lần** trong `s` (chính nó). Nếu `w` còn là substring của từ khác thì xuất hiện **ít nhất 2 lần**.

Dấu cách đảm bảo không có "trận khớp" bắc cầu giữa hai từ (vì `w` không chứa dấu cách). Ví dụ không có dấu cách: `["ab", "c", "bc"]` → `"abcbc"` chứa `"bc"` 2 lần dù `"bc"` không nằm trong từ nào khác!

In [12]:
def string_matching_join(words: list[str]) -> list[str]:
    s = " ".join(words)
    return [w for w in words if s.count(w) >= 2]

for words, expected in examples:
    print(words, "->", string_matching_join(words))

bad = "".join(["ab", "c", "bc"])
print("không dấu cách:", repr(bad), "count('bc') =", bad.count("bc"), "-> sai!")
print("có dấu cách  :", string_matching_join(["ab", "c", "bc"]))

['mass', 'as', 'hero', 'superhero'] -> ['as', 'hero']
['leetcode', 'et', 'code'] -> ['et', 'code']
['blue', 'green', 'bu'] -> []
không dấu cách: 'abcbc' count('bc') = 2 -> sai!
có dấu cách  : ['c']


**Tại sao `count` không chồng lấn vẫn đúng?** Ta chỉ cần biết "có ≥ 2 lần" — lần thứ nhất trong một từ chứa `w` khác `w`, lần thứ hai là chính `w`. Hai lần này nằm ở **hai từ khác nhau** (ngăn cách bởi dấu cách) nên không chồng lấn ⇒ `count` đếm được cả hai.

## 7. Cách 4 — tự cài KMP để kiểm tra substring

`in` của Python đã rất nhanh, nhưng phỏng vấn đôi khi yêu cầu tự viết. **KMP** (Knuth–Morris–Pratt) tìm `p` trong `s` trong `O(len(s) + len(p))`, không quay lui trên `s`.

**Mảng `lps`** (longest proper prefix which is also suffix): `lps[i]` = độ dài tiền tố dài nhất của `p[0..i]` đồng thời là hậu tố (không tính cả chuỗi). Khi so khớp thất bại ở ký tự `p[k]`, ta không bắt đầu lại từ 0 mà nhảy `k = lps[k − 1]`.

In [13]:
def build_lps(p: str) -> list[int]:
    lps = [0] * len(p)
    k = 0                                   # độ dài tiền tố-hậu tố hiện tại
    for i in range(1, len(p)):
        while k > 0 and p[i] != p[k]:
            k = lps[k - 1]
        if p[i] == p[k]:
            k += 1
        lps[i] = k
    return lps

for p in ["abab", "aaaa", "abcab", "hero"]:
    print(f"lps({p!r}) = {build_lps(p)}")

lps('abab') = [0, 0, 1, 2]
lps('aaaa') = [0, 1, 2, 3]
lps('abcab') = [0, 0, 0, 1, 2]
lps('hero') = [0, 0, 0, 0]


In [14]:
def kmp_contains(p: str, s: str) -> bool:
    if not p:
        return True
    lps = build_lps(p)
    k = 0                                   # số ký tự của p đã khớp
    for ch in s:
        while k > 0 and ch != p[k]:
            k = lps[k - 1]                  # lùi k, không lùi trên s
        if ch == p[k]:
            k += 1
            if k == len(p):
                return True
    return False

def string_matching_kmp(words: list[str]) -> list[str]:
    return [w for w in words if any(w != o and len(w) < len(o) and kmp_contains(w, o) for o in words)]

for words, expected in examples:
    print(words, "->", string_matching_kmp(words))

['mass', 'as', 'hero', 'superhero'] -> ['as', 'hero']
['leetcode', 'et', 'code'] -> ['et', 'code']
['blue', 'green', 'bu'] -> []


## 8. TRACE từng bước

Trace cách sắp theo độ dài với `words = ["leetcode", "et", "code"]`:

Sau sắp: `ws = ["et", "code", "leetcode"]`.

| i | ws[i] | j | ws[j] | `ws[i] in ws[j]`? | hành động | result |
|---:|---|---:|---|:---:|---|---|
| 0 | `et` | 1 | `code` | ✘ | thử tiếp | `[]` |
| 0 | `et` | 2 | `leetcode` | ✔ | thêm, break | `["et"]` |
| 1 | `code` | 2 | `leetcode` | ✔ | thêm, break | `["et","code"]` |
| 2 | `leetcode` | — | — | — | không còn từ dài hơn | `["et","code"]` |

In [15]:
def trace_string_matching(words: list[str]) -> list[str]:
    ws = sorted(words, key=len)
    print("sau khi sắp theo độ dài:", ws)
    result = []
    for i in range(len(ws)):
        found = False
        for j in range(i + 1, len(ws)):
            hit = ws[i] in ws[j]
            print(f"  i={i} {ws[i]!r:<11} j={j} {ws[j]!r:<12} in? {hit}")
            if hit:
                result.append(ws[i])
                found = True
                break
        if not found:
            print(f"  i={i} {ws[i]!r:<11} -> không có chuỗi cha")
        print(f"  result = {result}")
    return result

trace_string_matching(["leetcode", "et", "code"])

sau khi sắp theo độ dài: ['et', 'code', 'leetcode']
  i=0 'et'        j=1 'code'       in? False
  i=0 'et'        j=2 'leetcode'   in? True
  result = ['et']
  i=1 'code'      j=2 'leetcode'   in? True
  result = ['et', 'code']
  i=2 'leetcode'  -> không có chuỗi cha
  result = ['et', 'code']


['et', 'code']

Trace KMP khi tìm `p = "abab"` trong `s = "abaabab"` để thấy bước **lùi `k`** (thay vì quay lại đầu `s`):

In [16]:
def trace_kmp(p: str, s: str) -> bool:
    lps = build_lps(p)
    print(f"p={p!r}, lps={lps}, s={s!r}")
    k = 0
    for idx, ch in enumerate(s):
        while k > 0 and ch != p[k]:
            print(f"   s[{idx}]={ch!r} != p[{k}]={p[k]!r} -> k = lps[{k - 1}] = {lps[k - 1]}")
            k = lps[k - 1]
        if ch == p[k]:
            k += 1
        print(f"  s[{idx}]={ch!r}: đã khớp k={k} ký tự ({p[:k]!r})")
        if k == len(p):
            print("  => tìm thấy, kết thúc tại vị trí", idx)
            return True
    return False

trace_kmp("abab", "abaabab")

p='abab', lps=[0, 0, 1, 2], s='abaabab'
  s[0]='a': đã khớp k=1 ký tự ('a')
  s[1]='b': đã khớp k=2 ký tự ('ab')
  s[2]='a': đã khớp k=3 ký tự ('aba')
   s[3]='a' != p[3]='b' -> k = lps[2] = 1
   s[3]='a' != p[1]='b' -> k = lps[0] = 0
  s[3]='a': đã khớp k=1 ký tự ('a')
  s[4]='b': đã khớp k=2 ký tự ('ab')
  s[5]='a': đã khớp k=3 ký tự ('aba')
  s[6]='b': đã khớp k=4 ký tự ('abab')
  => tìm thấy, kết thúc tại vị trí 6


True

## 9. Code nộp LeetCode

In [17]:
class Solution:
    def stringMatching(self, words: list[str]) -> list[str]:
        ws = sorted(words, key=len)
        result = []
        for i in range(len(ws)):
            for j in range(i + 1, len(ws)):
                if ws[i] in ws[j]:
                    result.append(ws[i])
                    break
        return result

print([Solution().stringMatching(w) for w, _ in examples])

[['as', 'hero'], ['et', 'code'], []]


### Bản "nối rồi đếm" (1 dòng ý tưởng)

In [18]:
class SolutionJoin:
    def stringMatching(self, words: list[str]) -> list[str]:
        s = " ".join(words)
        return [w for w in words if s.count(w) >= 2]

print([SolutionJoin().stringMatching(w) for w, _ in examples])

[['as', 'hero'], ['et', 'code'], []]


### Bảng so sánh

Gọi `n` = số từ, `L` = độ dài tối đa, `S = n·L` = tổng độ dài.

| Cách | Ý tưởng | Time (xấu nhất) | Space | Ghi chú |
|---|---|---|---|---|
| 1. Vét cạn mọi cặp | `w in other` với mọi `other` | `O(n² · L²)` | `O(1)` phụ | đủ nhanh cho đề |
| 2. Sắp theo độ dài | chỉ thử với từ dài hơn | `O(n log n + n² · L²)`, ~½ số cặp | `O(n)` | sạch, không cần `i != j` |
| 3. Nối + `count` | xuất hiện ≥ 2 lần | `O(n · S · L)` | `O(S)` | ngắn gọn, cần dấu cách |
| 4. KMP | so khớp tuyến tính | `O(n² · L)` | `O(L)` | hiểu thuật toán chuỗi |

(`in`/`count` của CPython dùng thuật toán tìm kiếm tối ưu; trong thực tế thường nhanh hơn KMP viết bằng Python.)

## 10. Vì sao đúng?

- **Đủ:** nếu `w` là substring của từ khác `o` thì `len(w) < len(o)` (do các từ khác nhau). Sau khi sắp theo độ dài, `o` đứng **sau** `w` (với thứ tự ổn định, các từ dài hơn luôn đứng sau) ⇒ vòng `j > i` sẽ gặp `o` ⇒ `w` được thêm.
- **Không thừa:** `w` chỉ được thêm khi tìm thấy `ws[j]` (với `j != i`) chứa nó ⇒ đúng định nghĩa.
- **Không trùng:** `break` ngay sau khi thêm ⇒ mỗi `w` thêm tối đa một lần.
- **Cách nối:** `count(w) >= 2` ⇔ có lần xuất hiện ngoài vị trí của chính `w` ⇔ (do dấu cách không thuộc từ nào) lần đó nằm trọn trong một từ khác.

## 11. Time / Space

- Baseline: `n²` cặp × `O(L²)` mỗi lần `in` (xấu nhất với cài đặt ngây thơ) ⇒ `O(n²L²)`; với `n = 100`, `L = 30` khoảng `9·10^6` phép cơ bản — dư sức.
- Sắp xếp: thêm `O(n log n)`, số cặp giảm còn `n(n−1)/2`.
- KMP: mỗi cặp `O(L)` ⇒ `O(n²L)`.
- Bộ nhớ: `O(n)` cho bản sắp xếp và kết quả; `O(nL)` cho chuỗi nối.

In [19]:
import random, string, time
random.seed(1)
big = list({''.join(random.choice("ab") for _ in range(random.randint(1, 30))) for _ in range(300)})[:100]
for f in [string_matching_bruteforce, string_matching_sorted, string_matching_join, string_matching_kmp]:
    t0 = time.perf_counter()
    r = f(big)
    print(f"{f.__name__:<28} {len(r):>3} kết quả, {1000 * (time.perf_counter() - t0):7.2f} ms")

string_matching_bruteforce    30 kết quả,    0.58 ms
string_matching_sorted        30 kết quả,    0.38 ms
string_matching_join          30 kết quả,    0.76 ms
string_matching_kmp           30 kết quả,   10.57 ms


`string.ascii_lowercase` = `"abcdefghijklmnopqrstuvwxyz"` (dùng ở test bên dưới). Bảng chữ `"ab"` nhỏ giúp tạo nhiều quan hệ chứa nhau để test kỹ.

## 12. Bộ test

Mẫu đề; 1 từ; một từ chứa nhiều từ; chuỗi lồng nhiều tầng (`a ⊂ ab ⊂ abc`); subsequence nhưng không substring; từ nằm ở cuối/đầu chuỗi cha; và **đối chiếu ngẫu nhiên** mọi cách (so sánh dạng `sorted`).

In [20]:
def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    print(f"{name:<26} PASS -> {sorted(actual)}")

sol = Solution()
check("ví dụ 1", sol.stringMatching(["mass", "as", "hero", "superhero"]), ["as", "hero"])
check("ví dụ 2", sol.stringMatching(["leetcode", "et", "code"]), ["et", "code"])
check("ví dụ 3", sol.stringMatching(["blue", "green", "bu"]), [])
check("một từ", sol.stringMatching(["abc"]), [])
check("lồng nhiều tầng", sol.stringMatching(["abc", "a", "ab"]), ["a", "ab"])
check("đầu và cuối chuỗi cha", sol.stringMatching(["start", "st", "rt"]), ["st", "rt"])
check("subsequence không tính", sol.stringMatching(["abc", "ac"]), [])
check("cùng độ dài", sol.stringMatching(["ab", "ba"]), [])

ví dụ 1                    PASS -> ['as', 'hero']
ví dụ 2                    PASS -> ['code', 'et']
ví dụ 3                    PASS -> []
một từ                     PASS -> []
lồng nhiều tầng            PASS -> ['a', 'ab']
đầu và cuối chuỗi cha      PASS -> ['rt', 'st']
subsequence không tính     PASS -> []
cùng độ dài                PASS -> []


In [21]:
methods = [string_matching_bruteforce, string_matching_sorted, string_matching_any,
           string_matching_join, string_matching_kmp, Solution().stringMatching, SolutionJoin().stringMatching]
random.seed(1408)
for _ in range(1500):
    pool = {''.join(random.choice("abc") for _ in range(random.randint(1, 6))) for _ in range(random.randint(1, 10))}
    words = list(pool)
    expected = sorted(string_matching_bruteforce(words))
    for f in methods:
        assert sorted(f(words)) == expected, (f, words)
for p in ["a", "ab", "aba", "abab", "zz"]:
    for s in ["", "a", "abaabab", "zzz", "bab"]:
        assert kmp_contains(p, s) == (p in s), (p, s)
print("Mọi cách khớp nhau trên 1500 bộ từ ngẫu nhiên; KMP khớp `in`")
print("bảng chữ cái:", string.ascii_lowercase)

Mọi cách khớp nhau trên 1500 bộ từ ngẫu nhiên; KMP khớp `in`
bảng chữ cái: abcdefghijklmnopqrstuvwxyz


## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên `i != j` → mọi từ đều "là substring" của chính nó

In [22]:
def wrong_self(words):
    return [w for w in words if any(w in o for o in words)]     # SAI: o có thể là chính w

print("sai:", wrong_self(["blue", "green", "bu"]), "| đúng:", string_matching_bruteforce(["blue", "green", "bu"]))

sai: ['blue', 'green', 'bu'] | đúng: []


### Lỗi 2: Quên `break` → một từ bị thêm nhiều lần

In [23]:
def wrong_no_break(words):
    result = []
    for i in range(len(words)):
        for j in range(len(words)):
            if i != j and words[i] in words[j]:
                result.append(words[i])          # SAI: không break
    return result

print("sai:", wrong_no_break(["a", "ab", "abc"]), "| đúng:", string_matching_bruteforce(["a", "ab", "abc"]))

sai: ['a', 'a', 'ab'] | đúng: ['a', 'ab']


### Lỗi 3: Hiểu nhầm thành subsequence

Tự viết "kiểm tra từng ký tự của `w` có trong `o` theo thứ tự" là kiểm tra **subsequence**, khiến `"bu"` bị coi là nằm trong `"blue"`.

In [24]:
def is_subsequence(p, s):
    it = iter(s)
    return all(ch in it for ch in p)             # mẹo: 'in' trên iterator tiêu thụ dần

def wrong_subsequence(words):
    return [w for w in words if any(w != o and is_subsequence(w, o) for o in words)]

print("sai:", wrong_subsequence(["blue", "green", "bu"]), "| đúng:", string_matching_sorted(["blue", "green", "bu"]))

sai: ['bu'] | đúng: []


`iter(s)` tạo **iterator**; `ch in it` tiêu thụ iterator cho tới khi gặp `ch` — nên các ký tự sau phải xuất hiện **sau** ký tự trước. Đây là cách kiểm tra subsequence ngắn gọn (để phân biệt, không dùng cho bài này).

### Lỗi 4: Nối không có dấu phân cách

In [25]:
def wrong_join(words):
    s = "".join(words)                           # SAI: khớp bắc cầu giữa 2 từ
    return [w for w in words if s.count(w) >= 2]

print("sai:", wrong_join(["ab", "c", "bc"]), "| đúng:", string_matching_join(["ab", "c", "bc"]))

sai: ['c', 'bc'] | đúng: ['c']


## 14. Mẹo nhận dạng pattern

- "Chuỗi A có nằm trong chuỗi B" ⇒ `A in B` (Python), hoặc KMP / Z-function nếu cần tự cài, hoặc rolling hash.
- "So sánh mọi cặp" với `n` nhỏ (≤ 100–1000) ⇒ hai vòng lặp là chấp nhận được; tìm cách **cắt tỉa** (độ dài, sắp xếp).
- Quan hệ "chứa" có tính **thứ tự theo độ dài** ⇒ sắp theo `len` để chỉ xét một chiều.
- Mẹo **nối với ký tự phân cách** biến bài "nhiều chuỗi" thành bài "một chuỗi".

## 15. Biến thể

1. Trả về các từ **không** là substring của từ nào khác (phần bù).
2. Với mỗi từ, đếm **bao nhiêu** từ khác chứa nó.
3. Trả về các cặp `(con, cha)`.
4. LeetCode 28 *Find the Index of the First Occurrence in a String* — KMP chính là lời giải `O(n + m)`.
5. LeetCode 1662/1668 và các bài "is subsequence" (LeetCode 392) để phân biệt substring/subsequence.

In [26]:
def not_substrings(words):
    inside = set(string_matching_sorted(words))
    return [w for w in words if w not in inside]

def count_containers(words):
    return {w: sum(1 for o in words if o != w and w in o) for w in words}

def first_occurrence(haystack, needle):         # LeetCode 28, bản dùng find
    return haystack.find(needle)

print(not_substrings(["mass", "as", "hero", "superhero"]))
print(count_containers(["a", "ab", "abc", "b"]))
print(first_occurrence("sadbutsad", "sad"), first_occurrence("leetcode", "leeto"))

['mass', 'superhero']
{'a': 2, 'ab': 1, 'abc': 0, 'b': 2}
0 -1


## 16. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** `words = ["abc", "bc", "c", "abd"]` — đáp án là gì? Nêu chuỗi cha của từng từ.

**Bài 2 (đoán output):** `"ab" in "cab"`, `"ac" in "abc"`, `"xyz".find("z")`, `"a a".count("a")`, `sorted(["ccc", "a", "bb"], key=len)`.

**Bài 3 (điền chỗ trống):** `exercise_string_matching(words)` — hai vòng lặp, có `i != j` và `break`.

**Bài 4 (tự viết):** `exercise_is_substring(p, s)` — không dùng `in`/`find`, so slicing tại mọi vị trí.

**Bài 5 (biến thể):** `exercise_pairs(words)` — trả list các cặp `(con, cha)` với `con` là substring của `cha` (`con != cha`), sắp xếp tăng dần.

**Bài 6 (mini challenge):** `exercise_maximal_words(words)` — trả về các từ **không** nằm trong bất kỳ từ nào khác, theo thứ tự gốc, dùng mẹo nối chuỗi.

In [27]:
def exercise_string_matching(words: list[str]) -> list[str]:
    # TODO 1: result = []
    # TODO 2: hai vòng lặp i, j; điều kiện i != j and words[i] in words[j]
    # TODO 3: append rồi break
    pass

def exercise_is_substring(p: str, s: str) -> bool:
    # TODO: thử mọi vị trí bắt đầu i trong range(len(s) - len(p) + 1)
    pass

# assert sorted(exercise_string_matching(["mass", "as", "hero", "superhero"])) == ["as", "hero"]
# assert exercise_is_substring("as", "mass") and not exercise_is_substring("bu", "blue")

In [28]:
def exercise_pairs(words: list[str]) -> list[tuple[str, str]]:
    # TODO
    pass

def exercise_maximal_words(words: list[str]) -> list[str]:
    # TODO
    pass

# assert exercise_pairs(["a", "ab", "b"]) == [("a", "ab"), ("b", "ab")]
# assert exercise_maximal_words(["mass", "as", "hero", "superhero"]) == ["mass", "superhero"]

## 17. Gợi ý 3 mức

**Bài 1** — (1) Kiểm tra từng từ ngắn. (2) `"c"` nằm trong `"abc"` và `"bc"`. (3) `"bc"` nằm trong `"abc"`; `"abd"` thì không ai chứa.

**Bài 3** — (1) Giống baseline mục 3. (2) `for i in range(len(words)): for j in range(len(words))`. (3) `if i != j and words[i] in words[j]: result.append(words[i]); break`.

**Bài 4** — (1) Nếu `len(p) > len(s)` thì `range` rỗng. (2) `s[i:i + len(p)] == p`. (3) Hết vòng mà chưa khớp → `False`.

**Bài 5** — (1) Hai vòng lặp, không `break` (muốn mọi cha). (2) `pairs.append((w, o))`. (3) `return sorted(pairs)` — tuple so sánh theo phần tử thứ nhất rồi thứ hai.

**Bài 6** — (1) `s = " ".join(words)`. (2) Từ "tối đại" xuất hiện đúng 1 lần. (3) `[w for w in words if s.count(w) == 1]`.

## 18. Đáp án tham khảo

**Bài 1:** `"bc"` (cha `"abc"`), `"c"` (cha `"abc"`, `"bc"`) ⇒ đáp án `["bc", "c"]` (thứ tự tùy ý).

**Bài 2:** `True`, `False`, `2`, `2`, `['a', 'bb', 'ccc']`.

In [29]:
def answer_string_matching(words):
    result = []
    for i in range(len(words)):
        for j in range(len(words)):
            if i != j and words[i] in words[j]:
                result.append(words[i])
                break
    return result

def answer_is_substring(p, s):
    for i in range(len(s) - len(p) + 1):
        if s[i:i + len(p)] == p:
            return True
    return False

def answer_pairs(words):
    pairs = []
    for w in words:
        for o in words:
            if w != o and w in o:
                pairs.append((w, o))
    return sorted(pairs)

def answer_maximal_words(words):
    s = " ".join(words)
    return [w for w in words if s.count(w) == 1]

assert sorted(answer_string_matching(["abc", "bc", "c", "abd"])) == ["bc", "c"]
assert ("ab" in "cab", "ac" in "abc", "xyz".find("z"), "a a".count("a")) == (True, False, 2, 2)
assert sorted(["ccc", "a", "bb"], key=len) == ["a", "bb", "ccc"]
assert sorted(answer_string_matching(["mass", "as", "hero", "superhero"])) == ["as", "hero"]
assert answer_is_substring("as", "mass") and not answer_is_substring("bu", "blue")
assert answer_pairs(["a", "ab", "b"]) == [("a", "ab"), ("b", "ab")]
assert answer_maximal_words(["mass", "as", "hero", "superhero"]) == ["mass", "superhero"]
random.seed(5)
for _ in range(500):
    words = list({''.join(random.choice("ab") for _ in range(random.randint(1, 5))) for _ in range(8)})
    assert sorted(answer_string_matching(words)) == sorted(string_matching_join(words))
    assert sorted(answer_maximal_words(words)) == sorted(not_substrings(words))
    for p in words:
        for s in words:
            assert answer_is_substring(p, s) == (p in s)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def stringMatching(self, words: list[str]) -> list[str]:
        ws = sorted(words, key=len)                 # ngắn trước, dài sau
        res = []
        for i in range(len(ws)):
            for j in range(i + 1, len(ws)):         # chỉ thử từ dài hơn
                if ws[i] in ws[j]:                  # substring liên tiếp
                    res.append(ws[i])
                    break                           # không thêm trùng
        return res
```

Mẹo 1 dòng: `s = " ".join(words); return [w for w in words if s.count(w) >= 2]`.

**Checklist:**
- [ ] Không so một từ với chính nó.
- [ ] Mỗi từ thêm **tối đa một lần** (`break` hoặc `any`).
- [ ] Substring (liên tiếp), không phải subsequence.
- [ ] Nếu nối chuỗi: phải có **ký tự phân cách** không xuất hiện trong từ.